In [ ]:
# ============================================================
#  📝 CONFIGURATION — Set your file name & Markdown content here
# ============================================================

FILE_NAME = "test_pdf"  # DOCX file name (without .docx)

MD_CONTENT = r"""
# Spring AOP — Interview Questions

---

## Section 1: Core Concepts

**Q1. What is AOP and what problem does it solve?**

AOP (Aspect-Oriented Programming) solves the problem of cross-cutting concerns — behaviors like logging, security, transaction management, and metrics that need to apply across many classes but don't belong to any single one. Without AOP, you either duplicate this code in every method (scattering) or bury it inside business logic (tangling). AOP lets you express "apply this behavior to every method matching this pattern" in one place, without touching the target classes.

---

**Q2. What is a cross-cutting concern? Give real examples.**

A cross-cutting concern is a behavior that spans multiple layers or classes of an application but is not core business logic. Examples: logging every service method entry and exit, checking security before every write operation, measuring execution time for performance monitoring, managing database transactions, and recording audit trails for all data changes. These behaviors cut across OrderService, UserService, PaymentService — they do not belong to any one of them.

---

**Q3. Explain the core AOP terminology: Join Point, Pointcut, Advice, and Aspect.**

A **Join Point** is a specific point in program execution where behavior can be inserted — in Spring AOP, this is always a method invocation on a Spring-managed bean. A **Pointcut** is a predicate that selects which join points to target — for example, "all methods in the service package." **Advice** is the actual code that runs at a matched join point — the logging statement or security check. An **Aspect** is a class that combines pointcuts with advice methods, encapsulating one complete cross-cutting concern.

---

**Q4. What is the difference between the Target Object and the Proxy?**

The target object is the original bean — your actual `OrderService` class. The proxy is a wrapper Spring generates around it. When another bean requests `OrderService` through dependency injection, Spring gives them the proxy, not the target. All method calls go through the proxy first, which runs the advice chain, then delegates to the target. The target has no knowledge that it is being proxied.

---

**Q5. What is Weaving and when does Spring AOP perform it?**

Weaving is the process of linking aspects with their target objects to create advised proxies. Spring AOP performs weaving at **runtime** during application context startup — it creates proxy objects for beans whose methods match any pointcut. No bytecode modification happens. This is different from full AspectJ which can weave at compile time or class-load time by modifying the actual `.class` files.

---

## Section 2: Proxies — How It Works Internally

**Q6. How does Spring AOP work internally using proxies?**

During startup, a `BeanPostProcessor` called `AnnotationAwareAspectJAutoProxyCreator` scans every bean and evaluates all pointcut expressions from all `@Aspect` classes. If any pointcut matches a bean's methods, Spring creates a proxy wrapping that bean and registers the proxy in the context. When a method is called on the proxy, it builds an interceptor chain — an ordered list of advice methods that match the join point — executes them in sequence, and the target method is the final step. The return value travels back through the chain.

---

**Q7. What is the difference between JDK Dynamic Proxy and CGLIB proxy?**

JDK Dynamic Proxy creates a proxy that implements the target's interfaces. It can only intercept methods defined in those interfaces — you must inject by interface type, not concrete class. CGLIB generates a subclass of the target class at runtime using bytecode generation. It can intercept any non-final, non-private method and works even without interfaces. Since Spring Boot 2.0, CGLIB is the default. The constraint for CGLIB is that the target class and its methods cannot be `final`.

---

**Q8. When would a JDK proxy fail but a CGLIB proxy succeed?**

If you inject a bean by its concrete class type rather than an interface, JDK proxy fails with `BeanNotOfRequiredTypeException` at startup — the proxy only implements the interface, so it is not assignment-compatible with the concrete class. CGLIB succeeds because the proxy extends the concrete class and is therefore assignment-compatible with it.

```java
// JDK proxy — FAILS at startup
@Autowired
private OrderServiceImpl orderService; // proxy doesn't extend the impl class

// CGLIB — works fine
@Autowired
private OrderServiceImpl orderService; // proxy is a subclass of OrderServiceImpl
```

---

**Q9. What are the limitations of Spring AOP that come from using runtime proxies?**

Five key limitations. First, only method execution join points are supported — no field access, no constructor interception. Second, only Spring-managed beans can be advised — objects created with `new` are never proxied. Third, self-invocation bypasses the proxy (explained in detail later). Fourth, `final` classes and `final` or `private` methods cannot be advised with CGLIB. Fifth, static methods are never intercepted. Full AspectJ with bytecode weaving removes all these restrictions.

---

**Q10. What is the difference between Spring AOP and full AspectJ?**

Spring AOP borrows AspectJ's annotation syntax and pointcut expression language but uses runtime proxying as its mechanism — not AspectJ's weaving engine. Spring AOP can only intercept method executions on Spring beans. Full AspectJ performs bytecode weaving at compile time or load time, modifying actual `.class` files. It supports all join point types — field access, constructor execution, static methods, non-bean objects, `final` classes. Use full AspectJ when you need to advise code that Spring AOP simply cannot reach.

---

## Section 3: Advice Types

**Q11. What are the five advice types in Spring AOP and when do you use each?**

`@Before` — runs before the method. Use for pre-checks like authorization, parameter validation, or MDC setup. Cannot stop the method from running through normal return flow (throw an exception to block it).

`@After` — runs after the method regardless of outcome, like `finally`. Use for cleanup — clearing thread-local context, releasing resources. Cannot access the return value or exception.

`@AfterReturning` — runs only on successful return. Use for post-processing or audit logging of successful operations. Can inspect but not replace the return value.

`@AfterThrowing` — runs only when an exception is thrown. Use to observe and log exceptions. Cannot suppress the exception — it always propagates.

`@Around` — wraps the entire execution. Use when you need to span both sides (timing), control whether the method runs (caching, retry), or modify the return value. Must call `proceed()` and return the result.

---

**Q12. When should you use `@Around` vs `@Before`?**

Use the least powerful advice type that satisfies the requirement. `@Before` is sufficient when you only need to run logic before the method and have no need to affect the outcome. `@Around` is necessary when you need to measure time (requires spanning entry and exit), implement caching or retry (requires controlling whether `proceed()` is called), modify the return value, or handle exceptions and translate them. Defaulting to `@Around` for everything is a common mistake — `@Before` and `@AfterReturning` are simpler and express intent more clearly.

---

**Q13. What happens if you forget to call `proceed()` in an `@Around` advice?**

The target method never executes. The advice must return a value compatible with the target method's return type. If the method returns a primitive and the advice returns `null`, the JVM throws `NullPointerException` during unboxing. Skipping `proceed()` is sometimes intentional — in a caching aspect, if a cached value exists you return it directly without calling the target. But forgetting it accidentally is a silent bug where methods stop executing with no error.

---

**Q14. Can `@AfterThrowing` suppress or replace an exception?**

No. `@AfterThrowing` is an observation hook — the exception always propagates after the advice runs. If you throw a different exception from inside an `@AfterThrowing` method, that new exception does propagate, but this is considered abuse of the annotation's intent. The correct way to translate exceptions is with `@Around` — wrap `proceed()` in a try-catch, catch the original exception, and throw the translated one.

```java
// Correct — exception translation with @Around
@Around("within(com.example.repository..*)")
public Object translate(ProceedingJoinPoint pjp) throws Throwable {
    try {
        return pjp.proceed();
    } catch (DataAccessException ex) {
        throw new RepositoryException("DB operation failed", ex);
    }
}
```

---

**Q15. What is the difference between `@After` and `@AfterReturning`?**

`@After` runs regardless of outcome — success or exception. It is like `finally`. `@AfterReturning` only runs when the method returns normally with no exception. `@After` cannot access the return value. `@AfterReturning` can bind and inspect the return value using the `returning` attribute, and can even filter by return type — declaring `String result` means the advice only fires when the method returns a `String`.

---

## Section 4: Pointcut Expressions

**Q16. What does `execution(* com.example.service..*.*(..))` mean? Break it down.**

`execution` — matches method execution join points. `*` — any return type. `com.example.service..*` — any class in `com.example.service` or any sub-package (`..` means "and sub-packages"). `.*` — any method name. `(..)` — any number of parameters of any type. Together: intercept every method in the service package and all its sub-packages, regardless of return type, method name, or parameters.

---

**Q17. What is the difference between `execution()` and `within()`?**

`execution()` matches based on the method signature — you can filter by return type, method name, and parameter types. It is precise and flexible. `within()` matches all methods in a type or package without inspecting the method signature at all. It is coarser. `within(com.example.service.*)` and `execution(* com.example.service.*.*(..))` often select the same join points, but `execution()` lets you add method-level filters that `within()` cannot express.

---

**Q18. What does `@annotation()` pointcut do and why is it useful?**

`@annotation()` matches methods that are annotated with a specific annotation. It gives developers explicit opt-in control — only methods they annotate are advised. This is cleaner than package-level matching when you want selective application. When written in lowercase with a parameter name (like `@annotation(retryable)`), it also binds the annotation instance to that parameter, so the advice can read the annotation's attributes.

```java
@Around("@annotation(retryable)")
public Object retry(ProceedingJoinPoint pjp, Retryable retryable) throws Throwable {
    int maxAttempts = retryable.maxAttempts(); // read from the annotation
    // ...
}
```

---

**Q19. What is the difference between `this()` and `target()` in pointcut expressions?**

`this()` matches when the **proxy** is an instance of the specified type. `target()` matches when the **target object** (the real bean, not the proxy) is an instance of the specified type. With CGLIB (default in Spring Boot), they behave the same because the proxy subclasses the target. With JDK proxies, `this(ConcreteClass)` silently fails — the JDK proxy only implements interfaces, so `instanceof ConcreteClass` is false. Rule: always use `target()` when matching by concrete class type to be safe across both proxy mechanisms.

---

**Q20. How do you combine multiple pointcut expressions?**

Use `&&` (and), `||` (or), and `!` (not). For reuse, define named pointcuts with `@Pointcut` on empty methods and reference them by name. This avoids duplicating expressions across aspects.

```java
@Pointcut("within(com.example.service..*)")
public void inServiceLayer() {}

@Pointcut("@annotation(com.example.annotation.Audited)")
public void audited() {}

@Pointcut("inServiceLayer() && audited()")
public void auditedServiceOperation() {}
```

When the package changes, you update one `@Pointcut` definition — not every advice method that uses it.

---

## Section 5: The JoinPoint API

**Q21. What information can you get from a `JoinPoint` object?**

`getSignature()` — returns method name, declaring class, return type, parameter names and types. Cast to `MethodSignature` for full reflective access including reading method-level annotations. `getArgs()` — returns the actual argument values passed at runtime. `getTarget()` — the real, unproxied bean instance. `getThis()` — the proxy object. `toShortString()` / `toLongString()` — readable descriptions of the join point for logging.

---

**Q22. What is `ProceedingJoinPoint` and how is it different from `JoinPoint`?**

`ProceedingJoinPoint` extends `JoinPoint` and is only available in `@Around` advice. It adds `proceed()` which invokes the next interceptor in the chain (or the target method if there are no more interceptors). The other advice types (`@Before`, `@After`, etc.) do not have `proceed()` because the framework controls when the target executes — those advice types run at fixed points and cannot affect whether the method runs.

---

**Q23. Can you modify method arguments inside an `@Around` advice? What are the risks?**

Yes, using `pjp.proceed(modifiedArgs)`. You get the original args with `pjp.getArgs()`, modify the array, and pass it back. The risk is that `proceed(Object[])` accepts `Object[]` with no compile-time type safety. If you pass the wrong number of elements or wrong types, you get `IllegalArgumentException` or `ClassCastException` at runtime only. Also, if someone refactors the target method's parameters, the aspect silently breaks. Use narrow pointcuts targeting specific methods and add integration tests that exercise the advice path.

---

## Section 6: Self-Invocation (Most Asked)

**Q24. What is the self-invocation problem in Spring AOP?**

When a method inside a Spring bean calls another method on the same object using `this`, the call bypasses the proxy entirely. The proxy and the target are two separate objects. Once execution is inside the target, `this` refers to the target — not the proxy. Any method called via `this` goes directly to the target with no advice applied. This is the most common AOP pitfall and the same reason `@Transactional` doesn't work when called internally.

---

**Q25. What are the solutions to self-invocation and which is best?**

Three solutions, ranked by preference.

**Best — refactor into a separate bean.** Move the internally-called method to a different service. Inter-bean calls always go through the proxy. This also follows Single Responsibility — if the inner method needs its own cross-cutting behavior, it probably deserves its own bean.

**Acceptable — `AopContext.currentProxy()`.** Gets the current proxy and casts to the same type. Works but couples business code to Spring AOP infrastructure. Requires `@EnableAspectJAutoProxy(exposeProxy = true)` and fails in unit tests without Spring context.

**Last resort — `@Lazy` self-injection.** Inject the bean into itself with `@Lazy` to avoid circular dependency. Confusing to read and fragile — if someone removes `@Lazy`, startup fails with a circular dependency error that gives no hint about AOP.

---

**Q26. Does `@Transactional` have the same self-invocation problem?**

Yes, because `@Transactional` is implemented through AOP. When `processOrder()` calls `saveOrder()` on the same object, and `saveOrder()` is annotated with `@Transactional`, no transaction is created for `saveOrder()` — the proxy is bypassed. This is one of the most common bugs in Spring applications. The fix is the same: move the `@Transactional` method to a separate bean, or put `@Transactional` on the outer calling method.

---

## Section 7: Ordering and Configuration

**Q27. How do you control the execution order of multiple aspects?**

Use `@Order(n)` on the aspect class. Lower numbers execute first — the lowest-ordered aspect wraps all others (it runs outermost in the chain). Without `@Order`, execution order between aspects is undefined and can differ between JVM runs. `@Order` controls ordering between aspects, not between advice methods within the same aspect.

```
// With @Order(1) on Security, @Order(2) on Logging:
Security.pre → Logging.pre → target → Logging.post → Security.post
```

---

**Q28. How does `@Transactional` interact with custom aspect ordering?**

`@Transactional` is itself AOP — it uses `TransactionInterceptor` which defaults to `Ordered.LOWEST_PRECEDENCE`, meaning it runs closest to the target (innermost). Your custom aspects with lower `@Order` values run outside the transaction. This matters for logging: if your logging aspect has `@Order(1)` and logs "operation complete" in `@AfterReturning`, it fires after the transaction has committed — correct. If it ran inside the transaction, the log could say "complete" but the transaction might still roll back.

---

**Q29. How do you enable Spring AOP in a Spring Boot application?**

Spring Boot auto-configures AOP automatically — you just need the `spring-boot-starter-aop` dependency on the classpath. In non-Boot Spring, you add `@EnableAspectJAutoProxy` to a configuration class. CGLIB proxying is the default in Spring Boot 2.0+. To switch to JDK proxies: `@EnableAspectJAutoProxy(proxyTargetClass = false)` or `spring.aop.proxy-target-class=false`.

---

## Section 8: Real-World Usage

**Q30. Name real use cases where AOP is the right solution.**

Logging method entry, exit, and exceptions across the service layer without modifying any service. Measuring execution time and recording metrics for specific methods annotated with `@Timed`. Security checks before operations annotated with `@RequiresRole`. Caching — check cache before `proceed()`, store result after. Retry logic — call `proceed()` multiple times on failure. Audit logging on write operations. Rate limiting on API methods. Transaction management (Spring does this itself with `@Transactional`).

---

**Q31. What are common mistakes developers make when writing aspects?**

Using `@Around` everywhere when `@Before` or `@AfterReturning` would be sufficient. Writing overly broad pointcuts like `execution(* *(..))` that accidentally advise Spring's internal beans. Forgetting to return the result from `@Around` — methods silently return `null`. Not handling the `throws Throwable` in `@Around` — if the target throws a checked exception and you don't declare it, it gets wrapped. Assuming `@AfterThrowing` can suppress exceptions. Not writing tests to verify the aspect fires (or doesn't fire) correctly.

---

**Q32. How do you test that an aspect is working correctly?**

Two levels. For unit testing, treat the aspect as a plain Java class — mock `JoinPoint` or `ProceedingJoinPoint` with Mockito and call the advice method directly, asserting the side effects. For integration testing, load the Spring context with `@SpringBootTest`, verify the bean is proxied with `AopUtils.isAopProxy(bean)`, call the method, and assert the expected side effects occurred. Also write negative tests — call methods that should NOT match the pointcut and verify the aspect did not fire. This catches overly broad expressions before they reach production.

```java
// Integration — verify the bean is actually proxied
@Test
void orderServiceShouldBeProxied() {
    assertTrue(AopUtils.isAopProxy(orderService));
}
```

---

**Q33. What is a named `@Pointcut` and why should you use it?**

A `@Pointcut` is an annotation placed on an empty method — the method body is never executed. It defines a reusable pointcut expression that other advice methods reference by name. The benefit is DRY — when the package structure changes or you need to refine the pointcut, you update one definition and every advice that references it gets the update automatically. Without named pointcuts, the same expression is copy-pasted across every advice method and becomes a maintenance problem.

---

**Q34. What is the `args()` pointcut designator and how is it different from `execution()` parameter matching?**

`execution()` matches based on the **declared** parameter types in the method signature — a compile-time check. `args()` matches based on the **runtime** types of actual argument values — it checks `instanceof` at the time of invocation. They overlap for non-polymorphic cases, but `args()` can match subclasses that `execution()` would miss. `args()` also binds argument values directly to advice parameters, removing the need to call `joinPoint.getArgs()`.

---


"""

In [ ]:
# ============================================================
#  📦 Install Dependencies
# ============================================================
import subprocess, sys

packages = ["markdown", "Pygments", "python-docx", "beautifulsoup4", "lxml"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All dependencies installed successfully!")

In [ ]:
# ============================================================
#  🎨 DOCX Styling Engine — MD → HTML → DOCX
#  Same preprocessing pipeline as PDF version, but outputs
#  a styled Word document using python-docx.
# ============================================================

import re
import markdown
from markdown.extensions import Extension
from markdown.preprocessors import Preprocessor
from pygments import highlight as pyg_highlight
from pygments.lexers import get_lexer_by_name, guess_lexer, TextLexer
from pygments.formatters import HtmlFormatter
from bs4 import BeautifulSoup, NavigableString, Tag
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn, nsdecls
from docx.oxml import parse_xml


# ── Color Constants (same as PDF version) ──
QUESTION_RE = re.compile(r'^Q\d')

BG       = 'FFF2CC'
YELLOW   = 'FFFF00'
H1_BG    = 'FFE680'
H1_BDR   = 'C4A84A'
CODE_BG  = '1E1E1E'
CODE_FG  = 'D4D4D4'
CODE_ACC = '007ACC'
TH_BG    = 'E6D270'
TD_EVEN  = 'FFF8E1'
TD_ODD   = 'FFEDAA'
TBL_BDR  = 'B8960F'
IC_BG    = 'E8E0C8'
IC_FG    = 'C7254E'
DARK     = '1A1206'
TEXT_CLR = '2C2417'
LINK_CLR = '8B6914'
QUOTE_BG = 'F5EFC6'
HR_CLR   = 'C4A84A'


# ============================================================
#  Preprocessor 0: Auto-insert blank lines before list starts
# ============================================================
class ListSpacingPreprocessor(Preprocessor):
    LIST_RE = re.compile(r'^(\s*)([-*+]|\d+[.)]) ')

    def run(self, lines):
        result = []
        for line in lines:
            if self.LIST_RE.match(line):
                prev = ''
                for j in range(len(result) - 1, -1, -1):
                    if result[j].strip():
                        prev = result[j]
                        break
                if prev and not self.LIST_RE.match(prev):
                    result.append('')
            result.append(line)
        return result


class ListSpacingExtension(Extension):
    def extendMarkdown(self, md):
        md.preprocessors.register(
            ListSpacingPreprocessor(md), 'list_spacing', 120
        )


# ============================================================
#  Preprocessor 1: Fenced code → Pygments inline-styled HTML
# ============================================================
class InlineCodeHighlightPreprocessor(Preprocessor):
    FENCED_RE = re.compile(
        r'^```([\w+-]*)[ \t]*\n(.*?\n)```[ \t]*$',
        re.MULTILINE | re.DOTALL
    )

    def run(self, lines):
        text = '\n'.join(lines)
        result = []
        last_end = 0

        for m in self.FENCED_RE.finditer(text):
            result.append(text[last_end:m.start()])
            lang = m.group(1).strip() or ''
            code = m.group(2)
            if code.endswith('\n'):
                code = code[:-1]

            try:
                lexer = get_lexer_by_name(lang) if lang else TextLexer()
            except Exception:
                try:
                    lexer = guess_lexer(code)
                except Exception:
                    lexer = TextLexer()

            fmt = HtmlFormatter(noclasses=True, style='monokai',
                                wrapcode=True, linenos=False)
            highlighted = pyg_highlight(code, lexer, fmt)
            result.append(f'\n\n{highlighted}\n\n')
            last_end = m.end()

        result.append(text[last_end:])
        return ''.join(result).split('\n')


class InlineCodeHighlightExtension(Extension):
    def extendMarkdown(self, md):
        md.preprocessors.register(
            InlineCodeHighlightPreprocessor(md), 'inline_code_highlight', 110
        )


# ============================================================
#  HTML → DOCX Converter
# ============================================================
class HtmlToDocxConverter:
    """Converts markdown HTML to a fully styled DOCX document."""

    def __init__(self):
        self.doc = Document()
        self._setup_page()
        self._setup_styles()

    # ── Document setup ──
    def _setup_page(self):
        s = self.doc.sections[0]
        s.page_height, s.page_width = Cm(29.7), Cm(21.0)
        s.top_margin  = Cm(2.5)
        s.bottom_margin = Cm(3.0)
        s.left_margin = Cm(2.0)
        s.right_margin = Cm(2.0)

        # Page background color
        self.doc.element.insert(0,
            parse_xml(f'<w:background {nsdecls("w")} w:color="{BG}"/>'))
        self.doc.settings.element.append(
            parse_xml(f'<w:displayBackgroundShape {nsdecls("w")}/>'))

        # Footer with page numbers
        footer = s.footer
        footer.is_linked_to_previous = False
        fp = footer.paragraphs[0]
        fp.alignment = WD_ALIGN_PARAGRAPH.CENTER
        run = fp.add_run()
        run.font.size = Pt(8)
        run.font.color.rgb = RGBColor.from_string('8B7D6B')
        fld_xml = (
            f'<w:r {nsdecls("w")}>'
            f'<w:rPr><w:sz w:val="16"/><w:color w:val="8B7D6B"/></w:rPr>'
            f'<w:t xml:space="preserve">Page </w:t></w:r>'
            f'<w:r {nsdecls("w")}><w:rPr><w:sz w:val="16"/><w:color w:val="8B7D6B"/></w:rPr>'
            f'<w:fldChar w:fldCharType="begin"/></w:r>'
            f'<w:r {nsdecls("w")}><w:rPr><w:sz w:val="16"/><w:color w:val="8B7D6B"/></w:rPr>'
            f'<w:instrText> PAGE </w:instrText></w:r>'
            f'<w:r {nsdecls("w")}><w:rPr><w:sz w:val="16"/><w:color w:val="8B7D6B"/></w:rPr>'
            f'<w:fldChar w:fldCharType="end"/></w:r>'
            f'<w:r {nsdecls("w")}><w:rPr><w:sz w:val="16"/><w:color w:val="8B7D6B"/></w:rPr>'
            f'<w:t xml:space="preserve"> of </w:t></w:r>'
            f'<w:r {nsdecls("w")}><w:rPr><w:sz w:val="16"/><w:color w:val="8B7D6B"/></w:rPr>'
            f'<w:fldChar w:fldCharType="begin"/></w:r>'
            f'<w:r {nsdecls("w")}><w:rPr><w:sz w:val="16"/><w:color w:val="8B7D6B"/></w:rPr>'
            f'<w:instrText> NUMPAGES </w:instrText></w:r>'
            f'<w:r {nsdecls("w")}><w:rPr><w:sz w:val="16"/><w:color w:val="8B7D6B"/></w:rPr>'
            f'<w:fldChar w:fldCharType="end"/></w:r>'
        )
        for r_xml in re.findall(r'<w:r [^>]*>.*?</w:r>', fld_xml, re.DOTALL):
            fp._element.append(parse_xml(r_xml))

    def _setup_styles(self):
        st = self.doc.styles['Normal']
        st.font.name = 'Arial'
        st.font.size = Pt(11)
        st.font.color.rgb = RGBColor.from_string(TEXT_CLR)
        st.paragraph_format.space_after = Pt(8)
        st.paragraph_format.line_spacing = 1.75

    # ── Public entry point ──
    def convert(self, md_text):
        html = markdown.markdown(md_text, extensions=[
            ListSpacingExtension(),
            InlineCodeHighlightExtension(),
            'tables', 'toc', 'sane_lists', 'smarty', 'fenced_code',
        ])
        soup = BeautifulSoup(f'<body>{html}</body>', 'html.parser')
        self._walk(soup.body)
        return self.doc

    # ── Tree walker ──
    def _walk(self, el):
        for c in el.children:
            if isinstance(c, NavigableString):
                t = str(c).strip()
                if t:
                    p = self.doc.add_paragraph()
                    p.add_run(t)
            elif isinstance(c, Tag):
                self._tag(c)

    def _tag(self, t):
        n = t.name
        if   n == 'h1':       self._h1(t)
        elif n == 'h2':       self._h2(t)
        elif n in ('h3','h4','h5','h6'): self._hn(t, int(n[1]))
        elif n == 'p':        self._para(t)
        elif n in ('ul','ol'):self._list(t, n == 'ol')
        elif n == 'table':    self._table(t)
        elif n == 'blockquote': self._bq(t)
        elif n == 'hr':       self._hr()
        elif n == 'div' and 'highlight' in (t.get('class') or []):
            self._code_block(t)
        elif n == 'pre':      self._code_block(t)
        else:                 self._walk(t)

    # ── H1: centered, gold background, border ──
    def _h1(self, t):
        p = self.doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        pr = p._element.get_or_add_pPr()
        pr.append(parse_xml(
            f'<w:shd {nsdecls("w")} w:fill="{H1_BG}" w:val="clear"/>'))
        pr.append(parse_xml(
            f'<w:pBdr {nsdecls("w")}>'
            f'<w:top w:val="single" w:sz="18" w:space="6" w:color="{H1_BDR}"/>'
            f'<w:bottom w:val="single" w:sz="18" w:space="6" w:color="{H1_BDR}"/>'
            f'<w:left w:val="single" w:sz="18" w:space="6" w:color="{H1_BDR}"/>'
            f'<w:right w:val="single" w:sz="18" w:space="6" w:color="{H1_BDR}"/>'
            f'</w:pBdr>'))
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after = Pt(16)
        self._inline(p, t, bold=True, font_name='Georgia',
                     font_size=Pt(22), color=DARK)

    # ── H2: section heading ──
    def _h2(self, t):
        p = self.doc.add_paragraph()
        p.paragraph_format.space_before = Pt(16)
        p.paragraph_format.space_after = Pt(6)
        p._element.get_or_add_pPr().append(
            parse_xml(f'<w:shd {nsdecls("w")} w:fill="{BG}" w:val="clear"/>'))
        self._inline(p, t, bold=True, font_size=Pt(12.5), color=DARK)

    # ── H3–H6 ──
    def _hn(self, t, level):
        p = self.doc.add_paragraph()
        p.paragraph_format.space_before = Pt(12)
        p.paragraph_format.space_after = Pt(4)
        p._element.get_or_add_pPr().append(
            parse_xml(f'<w:shd {nsdecls("w")} w:fill="{BG}" w:val="clear"/>'))
        if level == 3:
            pr = p._element.get_or_add_pPr()
            pr.append(parse_xml(
                f'<w:pBdr {nsdecls("w")}>'
                f'<w:bottom w:val="single" w:sz="4" w:space="2" w:color="D4C9A8"/>'
                f'</w:pBdr>'))
        sz = {3: 11.5, 4: 11, 5: 10.5, 6: 10}.get(level, 11)
        self._inline(p, t, bold=True, font_size=Pt(sz), color='3D3019')

    # ── Paragraph ──
    def _para(self, t):
        p = self.doc.add_paragraph()
        p._element.get_or_add_pPr().append(
            parse_xml(f'<w:shd {nsdecls("w")} w:fill="{BG}" w:val="clear"/>'))
        self._inline(p, t)

    # ── Inline content processor (recursive) ──
    def _inline(self, p, el, **sty):
        for c in el.children:
            if isinstance(c, NavigableString):
                txt = str(c)
                if txt:
                    r = p.add_run(txt)
                    self._style_run(r, **sty)
            elif isinstance(c, Tag):
                ns = dict(sty)
                if c.name in ('strong', 'b'):
                    ns['bold'] = True
                    if QUESTION_RE.match(c.get_text().strip()):
                        ns['highlight'] = True
                elif c.name in ('em', 'i'):
                    ns['italic'] = True
                elif c.name == 'code':
                    ns['code'] = True
                elif c.name == 'a':
                    ns['link'] = True
                elif c.name == 'br':
                    p.add_run('\n')
                    continue
                elif c.name == 'span':
                    m = re.search(r'color:\s*#([0-9a-fA-F]{6})',
                                  c.get('style', ''))
                    if m:
                        ns['color'] = m.group(1)
                self._inline(p, c, **ns)

    def _style_run(self, r, bold=False, italic=False, code=False,
                   highlight=False, link=False, color=None,
                   font_name=None, font_size=None):
        if bold:     r.bold = True
        if italic:   r.italic = True
        if font_name: r.font.name = font_name
        if font_size: r.font.size = font_size
        if code:
            r.font.name = 'Consolas'
            if not font_size:
                r.font.size = Pt(10)
        if highlight:
            r._element.get_or_add_rPr().append(
                parse_xml(f'<w:shd {nsdecls("w")} w:fill="{YELLOW}" w:val="clear"/>'))
            if not color:
                r.font.color.rgb = RGBColor.from_string(DARK)
        elif code and not color:
            r.font.color.rgb = RGBColor.from_string(IC_FG)
            r._element.get_or_add_rPr().append(
                parse_xml(f'<w:shd {nsdecls("w")} w:fill="{IC_BG}" w:val="clear"/>'))
        elif link:
            r.font.color.rgb = RGBColor.from_string(LINK_CLR)
            r.font.underline = True
        if color:
            r.font.color.rgb = RGBColor.from_string(color)

    # ── Code block with Pygments token colors ──
    def _code_block(self, t):
        pre = t.find('pre') if t.name == 'div' else t
        if not pre:
            return
        p = self.doc.add_paragraph()
        pr = p._element.get_or_add_pPr()
        pr.append(parse_xml(
            f'<w:shd {nsdecls("w")} w:fill="{CODE_BG}" w:val="clear"/>'))
        pr.append(parse_xml(
            f'<w:pBdr {nsdecls("w")}>'
            f'<w:left w:val="single" w:sz="24" w:space="10" w:color="{CODE_ACC}"/>'
            f'</w:pBdr>'))
        p.paragraph_format.space_before = Pt(6)
        p.paragraph_format.space_after = Pt(10)
        p.paragraph_format.left_indent = Cm(0.5)
        p.paragraph_format.right_indent = Cm(0.5)
        p.paragraph_format.line_spacing = 1.7

        for desc in pre.descendants:
            if isinstance(desc, NavigableString):
                txt = str(desc)
                if not txt:
                    continue
                r = p.add_run(txt)
                r.font.name = 'Consolas'
                r.font.size = Pt(10)
                parent = desc.parent
                if parent and parent.name == 'span':
                    m = re.search(r'color:\s*#([0-9a-fA-F]{6})',
                                  parent.get('style', ''))
                    clr = m.group(1) if m else CODE_FG
                else:
                    clr = CODE_FG
                r.font.color.rgb = RGBColor.from_string(clr)

    # ── Table with gold headers, alternating rows ──
    def _table(self, t):
        rows = t.find_all('tr')
        if not rows:
            return
        ncols = len(rows[0].find_all(['th', 'td']))
        tbl = self.doc.add_table(rows=len(rows), cols=ncols)
        tbl.alignment = WD_TABLE_ALIGNMENT.CENTER

        # Table borders
        tblPr = tbl._element.find(qn('w:tblPr'))
        if tblPr is None:
            tblPr = parse_xml(f'<w:tblPr {nsdecls("w")}/>')
            tbl._element.insert(0, tblPr)
        tblPr.append(parse_xml(
            f'<w:tblBorders {nsdecls("w")}>'
            f'<w:top w:val="single" w:sz="12" w:color="{TBL_BDR}"/>'
            f'<w:bottom w:val="single" w:sz="12" w:color="{TBL_BDR}"/>'
            f'<w:left w:val="single" w:sz="12" w:color="{TBL_BDR}"/>'
            f'<w:right w:val="single" w:sz="12" w:color="{TBL_BDR}"/>'
            f'<w:insideH w:val="single" w:sz="6" w:color="{TBL_BDR}"/>'
            f'<w:insideV w:val="single" w:sz="6" w:color="{TBL_BDR}"/>'
            f'</w:tblBorders>'))

        # Full-width table
        tblPr.append(parse_xml(
            f'<w:tblW {nsdecls("w")} w:type="pct" w:w="5000"/>'))

        for i, row_tag in enumerate(rows):
            cells = row_tag.find_all(['th', 'td'])
            for j, cell_tag in enumerate(cells):
                if j >= ncols:
                    break
                cell = tbl.rows[i].cells[j]
                cell.text = ''
                p = cell.paragraphs[0]
                p.paragraph_format.space_before = Pt(4)
                p.paragraph_format.space_after = Pt(4)
                is_hdr = cell_tag.name == 'th'
                self._inline(p, cell_tag, bold=is_hdr, font_size=Pt(10.5))
                bg = TH_BG if is_hdr else (TD_EVEN if i % 2 == 0 else TD_ODD)
                self._cell_shading(cell, bg)

    def _cell_shading(self, cell, color):
        tc = cell._element
        tcPr = tc.find(qn('w:tcPr'))
        if tcPr is None:
            tcPr = parse_xml(f'<w:tcPr {nsdecls("w")}/>')
            tc.insert(0, tcPr)
        tcPr.append(parse_xml(
            f'<w:shd {nsdecls("w")} w:fill="{color}" w:val="clear"/>'))

    # ── Lists: ordered & unordered ──
    def _list(self, t, ordered=False, level=0):
        items = t.find_all('li', recursive=False)
        for idx, li in enumerate(items):
            p = self.doc.add_paragraph()
            p._element.get_or_add_pPr().append(
                parse_xml(f'<w:shd {nsdecls("w")} w:fill="{BG}" w:val="clear"/>'))
            indent = Cm(1.27 * (level + 1))
            p.paragraph_format.left_indent = indent
            p.paragraph_format.first_line_indent = Cm(-0.63)
            p.paragraph_format.space_before = Pt(2)
            p.paragraph_format.space_after = Pt(3)
            p.paragraph_format.line_spacing = 1.7

            # Bullet / number prefix
            if ordered:
                prefix = f'{idx + 1}. '
            else:
                prefix = ['•', '◦', '▪'][min(level, 2)] + ' '
            r = p.add_run(prefix)
            r.font.size = Pt(11)
            r.font.color.rgb = RGBColor.from_string(TEXT_CLR)

            # Content (skip nested lists — they're processed recursively)
            for c in li.children:
                if isinstance(c, NavigableString):
                    txt = str(c)
                    if txt.strip():
                        r = p.add_run(txt)
                        r.font.size = Pt(11)
                        r.font.color.rgb = RGBColor.from_string(TEXT_CLR)
                elif isinstance(c, Tag):
                    if c.name in ('ul', 'ol'):
                        self._list(c, c.name == 'ol', level + 1)
                    else:
                        self._inline(p, c)

    # ── Blockquote ──
    def _bq(self, t):
        for c in t.children:
            if isinstance(c, Tag) and c.name == 'p':
                p = self.doc.add_paragraph()
                pr = p._element.get_or_add_pPr()
                pr.append(parse_xml(
                    f'<w:pBdr {nsdecls("w")}>'
                    f'<w:left w:val="single" w:sz="24" w:space="8" w:color="{H1_BDR}"/>'
                    f'</w:pBdr>'))
                pr.append(parse_xml(
                    f'<w:shd {nsdecls("w")} w:fill="{QUOTE_BG}" w:val="clear"/>'))
                p.paragraph_format.left_indent = Cm(1)
                p.paragraph_format.space_before = Pt(6)
                p.paragraph_format.space_after = Pt(6)
                self._inline(p, c, italic=True, color='3D3019')
            elif isinstance(c, NavigableString):
                txt = str(c).strip()
                if txt:
                    p = self.doc.add_paragraph()
                    pr = p._element.get_or_add_pPr()
                    pr.append(parse_xml(
                        f'<w:shd {nsdecls("w")} w:fill="{QUOTE_BG}" w:val="clear"/>'))
                    p.paragraph_format.left_indent = Cm(1)
                    r = p.add_run(txt)
                    r.italic = True

    # ── Horizontal rule ──
    def _hr(self):
        p = self.doc.add_paragraph()
        pr = p._element.get_or_add_pPr()
        pr.append(parse_xml(
            f'<w:pBdr {nsdecls("w")}>'
            f'<w:bottom w:val="single" w:sz="12" w:space="1" w:color="{HR_CLR}"/>'
            f'</w:pBdr>'))
        p.paragraph_format.space_before = Pt(12)
        p.paragraph_format.space_after = Pt(12)


print("✅ DOCX styling engine loaded")

In [ ]:
# ============================================================
#  🔨 Generate DOCX from Markdown
# ============================================================

import os
from pathlib import Path
from IPython.display import display, HTML as IPHTML

# Detect environment and set output dir
if os.name == 'nt':
    OUTPUT_DIR = str(Path.home() / "Downloads")
elif os.path.isdir("/content"):
    OUTPUT_DIR = "/content"
else:
    OUTPUT_DIR = str(Path.home() / "Downloads")

os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_docx(file_name, md_content):
    safe_name = re.sub(r'[^\w\s-]', '', file_name).strip().replace(' ', '_') or "document"
    docx_path = os.path.join(OUTPUT_DIR, f"{safe_name}.docx")

    converter = HtmlToDocxConverter()
    doc = converter.convert(md_content)
    doc.save(docx_path)

    size_kb = os.path.getsize(docx_path) / 1024
    print(f"✅ DOCX generated: {docx_path} ({size_kb:.1f} KB)")
    return docx_path

docx_file = generate_docx(FILE_NAME, MD_CONTENT)

In [ ]:
# ============================================================
#  ⬇️ Download Button
# ============================================================

import base64

with open(docx_file, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

fname = os.path.basename(docx_file)

display(IPHTML(f"""
<div style="text-align:center; margin:20px 0;">
    <a href="data:application/vnd.openxmlformats-officedocument.wordprocessingml.document;base64,{b64}" download="{fname}"
       style="
        display:inline-block; padding:14px 40px;
        background-color:#2b579a; color:#fff;
        font-family:Arial,sans-serif; font-size:15px; font-weight:700;
        text-decoration:none; border-radius:8px;
        box-shadow:0 3px 10px rgba(0,0,0,0.15); cursor:pointer;
       ">⬇ Download {fname}</a>
</div>
"""))